# 행렬식은 넓이다

> 선형대수 12강 · 행렬식

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [행렬식은 넓이다](https://mioon1402.github.io/timeseriesdata/linalg/L12-determinant.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 넓이부터 다시

## 1. 평행사변형을 직접 찌그러뜨리기

## 2. 부호는 무엇을 뜻하나

## 3. 행렬식을 정의하는 세 가지 규칙

## 4. 거기서 나오는 성질들

## 5. 계산법 — 여인수 전개와 크라머 공식

## 6. numpy 로 확인하기

**12-1. 행렬식과 넓이**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[3., 1.],
              [1., 2.]])

print("det =", np.linalg.det(A))
print("손계산 ad - bc =", 3*2 - 1*1)
print()
print("→ 두 열 (3,1), (1,2) 이 만드는 평행사변형의 넓이가 5")

**12-2. 성질 확인**

In [ ]:
B = np.array([[2., 1.],
              [1., 2.]])

dA, dB = np.linalg.det(A), np.linalg.det(B)

print(f"det(A)     = {dA:.4f}")
print(f"det(B)     = {dB:.4f}")
print(f"det(AB)    = {np.linalg.det(A @ B):.4f}   vs  det(A)·det(B) = {dA*dB:.4f}")
print(f"det(Aᵀ)    = {np.linalg.det(A.T):.4f}")
print(f"det(A⁻¹)   = {np.linalg.det(np.linalg.inv(A)):.4f}   vs  1/det(A) = {1/dA:.4f}")
print(f"det(2A)    = {np.linalg.det(2*A):.4f}   vs  2²·det(A) = {4*dA:.4f}")
print()
print(f"det(A+B)   = {np.linalg.det(A + B):.4f}   vs  det(A)+det(B) = {dA+dB:.4f}")
print("            ↑ 덧셈은 성립하지 않는다!")

**12-3. 소거해도 넓이는 변하지 않는다**

In [ ]:
C = A.copy()
print("원본 det =", np.linalg.det(C))

C[1] -= (C[1, 0] / C[0, 0]) * C[0]      # 2행에서 1행의 배수를 뺀다 (3강)
print("소거 후 =")
print(C)
print("det =", np.linalg.det(C), " ← 그대로")
print()
print("삼각행렬이 되었으니 대각 원소의 곱으로도 확인:")
print("  ", C[0, 0], "×", round(C[1, 1], 4), "=", round(C[0, 0] * C[1, 1], 4))
print()
print("→ 평행사변형을 '기울여도' 밑변과 높이가 그대로라 넓이가 안 변한다")

**12-4. 여인수 전개 직접 구현**

In [ ]:
def 여인수_det(M):
    M = np.asarray(M, dtype=float)
    n = len(M)
    if n == 1:
        return M[0, 0]
    총합 = 0.0
    for j in range(n):
        소행렬 = np.delete(np.delete(M, 0, axis=0), j, axis=1)   # 0행과 j열 삭제
        총합 += ((-1) ** j) * M[0, j] * 여인수_det(소행렬)
    return 총합

E = np.array([[ 2., 1., 1.],
              [ 4., -6., 0.],
              [-2., 7., 2.]])

print("여인수 전개 =", 여인수_det(E))
print("numpy      =", round(np.linalg.det(E), 10))

**12-5. 크라머 공식 — 아름답지만 느리다**

In [ ]:
b = np.array([5., -2., 9.])

def 크라머(A, b):
    detA = np.linalg.det(A)
    해 = []
    for i in range(A.shape[1]):
        B_i = A.copy()
        B_i[:, i] = b                       # i번째 열을 b 로 교체
        해.append(np.linalg.det(B_i) / detA)
    return np.array(해)

print("크라머 :", np.round(크라머(E, b), 6))
print("solve  :", np.linalg.solve(E, b))
print()

import time, math
for n in [5, 10, 15, 20]:
    print(f"n={n:2d}  여인수 전개 {math.factorial(n):>22,}회  vs  소거법 {n**3:>8,}회")

**12-6. det 로 특이성을 판정하면 안 되는 이유**

In [ ]:
# 완벽하게 멀쩡한 단위행렬을 100배 줄인다
좋은행렬 = 0.01 * np.eye(20)

print("0.01·I(20×20)")
print("  det       =", np.linalg.det(좋은행렬), "  ← 0 처럼 보인다!")
print("  조건수    =", np.linalg.cond(좋은행렬), "  ← 1. 완벽한 행렬")
print("  랭크      =", np.linalg.matrix_rank(좋은행렬), "/ 20")
print()

# 거의 특이한 행렬
나쁜행렬 = np.array([[1., 1.], [1., 1 + 1e-12]])
print("[[1,1],[1,1+1e-12]]")
print("  det       =", np.linalg.det(나쁜행렬))
print("  조건수    =", f"{np.linalg.cond(나쁜행렬):.3e}", "  ← 사실상 특이")
print()
print("→ det 의 크기는 '특이한 정도' 를 말해주지 않는다. 조건수를 보세요.")

**12-7. 연습문제**

In [ ]:
# 문제 1. [[1,2],[3,4]] 의 행렬식을 손으로 구하고 확인하세요.

# 문제 2. 두 열을 바꾸면 부호가 뒤집히는지 확인하세요.

# 문제 3. 3×3 단위행렬의 한 행에 5를 곱하면 det 는?
#         세 행 전부에 5를 곱하면?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1 — 1·4 − 2·3 = −2
A1 = np.array([[1., 2.], [3., 4.]])
print("문제 1: 손계산 -2, numpy", round(np.linalg.det(A1), 10))
print("        음수 = 두 열의 배향이 시계 방향")

# 문제 2
A2 = A1[:, [1, 0]]
print("\n문제 2: 열 교환 후 det =", round(np.linalg.det(A2), 10), " ← 부호만 뒤집힘")

# 문제 3
I3 = np.eye(3)
한행 = I3.copy(); 한행[0] *= 5
전부 = 5 * I3
print("\n문제 3: 한 행만 5배 →", round(np.linalg.det(한행), 6), " (5¹)")
print("        세 행 전부 5배 →", round(np.linalg.det(전부), 6), " (5³ = 125)")
print("        det(cA) = cⁿ det(A) 이기 때문")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)